# 20 peaks

In [1]:
import numpy as np

In [ ]:
38394, 38449, 38944, 41251, 41570, 42057, 43139, 43170, 43245, 43961, 44955, 45000, 45768, 45809, 46738, 47454, 47650, 49911

peaks_index = np.array([38394, 38944, 41570, 43139, 43961, 46738, 47454, 49911])
weak_index = np.array([38449, 41251, 42057, 42069, 42085, 43170, 43221, 43245, 44955, 45000, 45768, 45809, 47650 ])


42069, 42085, 43221

In [ ]:
import numpy as np
peaks_index = np.array([38394, 38944, 41570, 43139, 43961, 46738, 47454, 49911])
weak_index = np.array([38449, 41251, 42057, 42069, 42085, 43170, 43221, 43245, 44955, 45000, 45768, 45809, 47650 ])



In [ ]:
combined = np.sort(np.concatenate([peaks_index, weak_index]))
print("Combined sorted indices:", combined)

Combined sorted indices: [38394 38449 38944 41251 41570 42057 42069 42085 43139 43170 43221 43245
 43961 44955 45000 45768 45809 46738 47454 47650 49911]


In [ ]:
np.diff(combined)

array([  55,  495, 2307,  319,  487,   12,   16, 1054,   31,   51,   24,
        716,  994,   45,  768,   41,  929,  716,  196, 2261])

In [ ]:
len(combined)

21

# POP MD calibration

In [ ]:
import numpy as np
from slippage_clbrt_shift import shift_depth_piecewise, invert_shift_depth_piecewise

gauges = np.array([13004, 15321, 13753.1])
gauge_to_fiber = invert_shift_depth_piecewise(gauges, z0=12000.0, z1=16000.0, s0=145.0, s1=160.0)
print("测点深度 gauges: ", gauges)
print("对应纤维深度 gauge_to_fiber: ", gauge_to_fiber)


测点深度 gauges:  [13004.  15321.  13753.1]
对应纤维深度 gauge_to_fiber:  [12855.79078456 15164.13449564 13602.09215442]


In [ ]:
fibers = np.array([12660.6, 12678.7, 12842.0, 13602.7, 13707.9, 13868.5, 13872.4, 13877.7, 14225.3, 14235.5, 
                   14252.3, 14260.2, 14496.3, 14824.1, 14839.0, 15092.2, 15105.7, 15412.0, 15648.1, 15712.8, 16458.3])

fiber_to_gauge = shift_depth_piecewise(fibers, z0=12000.0, z1=16000.0, s0=145.0, s1=160.0)
print("纤维深度 fibers: ", fibers)
print("对应测点深度 fiber_to_gauge: ", fiber_to_gauge)


纤维深度 fibers:  [12660.6 12678.7 12842.  13602.7 13707.9 13868.5 13872.4 13877.7 14225.3
 14235.5 14252.3 14260.2 14496.3 14824.1 14839.  15092.2 15105.7 15412.
 15648.1 15712.8 16458.3]
对应测点深度 fiber_to_gauge:  [12808.07725  12826.245125 12990.1575   13753.710125 13859.304625
 14020.506875 14024.4215   14029.741375 14378.644875 14388.883125
 14405.746125 14413.67575  14650.661125 14979.690375 14994.64625
 15248.79575  15262.346375 15569.795    15806.780375 15871.723
 16618.3     ]


# POP channel to depth --> POP depth to flowback depth --> flowback depth to channel

In [1]:
import h5py
import numpy as np
import pandas as pd

# ===== 输入 =====
evo5_path = r'C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 5 (POP)/Zgabay A14H - pop - strain change.h5'
evo7_path = r'C:/Users/fengxiang.mao/Desktop/Devon data/HFTS-1_Devon/Evo_5_6_7_Sep_2025/Evo 7 (flowback)/corrected_within_1h_(with_data_wash).h5'

pop_channels = np.array([38394, 38449, 38944, 41251, 41570, 42057, 42069, 42085, 43139, 43170,
                         43221, 43245, 43961, 44955, 45000, 45768, 45809, 46738, 47454, 47650, 49911], dtype=int)

# ===== Step 1: 在 Evo5 depth 中找到对应深度 POP_depth =====
with h5py.File(evo5_path, "r") as f5:
    depth5 = np.asarray(f5["depth"], dtype=float)

# channel -> MD
POP_depth = depth5[pop_channels]

# ===== Step 2 & 3: 在 Evo7 depth 中找最接近 POP_depth 的 depth，并输出对应的 Evo7 channel =====
with h5py.File(evo7_path, "r") as f7:
    depth7 = np.asarray(f7["depth"], dtype=float)

# 安全检查：depth7 必须单调递增，否则 searchsorted 不可靠
if not np.all(np.diff(depth7) >= 0):
    raise ValueError("Evo7 depth 不是单调递增，不能直接用 searchsorted；需要先排序并保留索引映射。")

# searchsorted 找插入位置，然后和左边那个比谁更近
idx_right = np.searchsorted(depth7, POP_depth, side="left")
idx_left = np.clip(idx_right - 1, 0, len(depth7) - 1)
idx_right = np.clip(idx_right, 0, len(depth7) - 1)

left_dist = np.abs(depth7[idx_left] - POP_depth)
right_dist = np.abs(depth7[idx_right] - POP_depth)

flowback_channel = np.where(right_dist < left_dist, idx_right, idx_left).astype(int)
flowback_depth = depth7[flowback_channel]
diff_ft = flowback_depth - POP_depth  # 正负表示flowback更深/更浅

# ===== 输出（表格更直观）=====
df = pd.DataFrame({
    "pop_channel": pop_channels,
    "POP_depth_ft": POP_depth,
    "flowback_channel": flowback_channel,
    "flowback_depth_ft": flowback_depth,
    "diff_ft": diff_ft
})

print(df.to_string(index=False))
print("\nSummary:")
print("max abs diff (ft):", np.max(np.abs(diff_ft)))
print("mean abs diff (ft):", np.mean(np.abs(diff_ft)))


 pop_channel  POP_depth_ft  flowback_channel  flowback_depth_ft   diff_ft
       38394  12660.612305             38271       12660.712891  0.100586
       38449  12678.748047             38326       12678.911133  0.163086
       38944  12841.974609             38819       12842.001953  0.027344
       41251  13602.713867             41119       13602.877930  0.164062
       41570  13707.903320             41436       13707.747070 -0.156250
       42057  13868.495117             41922       13868.525391  0.030273
       42069  13872.452148             41934       13872.494141  0.041992
       42085  13877.724609             41950       13877.788086  0.063477
       43139  14225.286133             43000       14225.144531 -0.141602
       43170  14235.508789             43031       14235.401367 -0.107422
       43221  14252.327148             43082       14252.273438 -0.053711
       43245  14260.240234             43106       14260.209961 -0.030273
       43961  14496.340820            

In [2]:
print(flowback_depth)

[12660.71289062 12678.91113281 12842.00195312 13602.87792969
 13707.74707031 13868.52539062 13872.49414062 13877.78808594
 14225.14453125 14235.40136719 14252.2734375  14260.20996094
 14496.41503906 14824.25488281 14838.80957031 15092.21484375
 15105.77832031 15412.11523438 15648.31835938 15712.82714844
 16458.48632812]
